In [0]:
# ============================================================
# NOTEBOOK 04 — SILVER: MODELO SparkML
# Vermont School - Sistema de Alerta Temprana 2025-26
# ============================================================

TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Leer desde Trusted
df = spark.read.parquet(f"{TRUSTED}/estudiantes_2025_26")
df.createOrReplaceTempView("estudiantes")
print(f"✓ Datos cargados: {df.count()} estudiantes")

# Features numéricas para el modelo
FEATURES = [
    'total_ausencias', 'ausencia_clase', 'ausencia_clase_excusa',
    'ausencia_colegio', 'retardo', 'salida_temprana',
    'n_f1', 'n_f2', 'n_tipos_falta',
    'promedio_acumulado', 'nota_min_acumulada'
]

# Verificar nulls en features
print("\nNulls en features:")
for f in FEATURES:
    n = df.filter(F.col(f).isNull()).count()
    if n > 0:
        print(f"  {f}: {n}")

# Rellenar nulls con 0
df_clean = df.fillna(0, subset=FEATURES)
print(f"✓ Nulls rellenados")
print(f"\nDistribución target (nivel_riesgo):")
df_clean.groupBy('nivel_riesgo').count().orderBy('nivel_riesgo').show()

In [0]:
# Configurar ruta temporal para SparkML en Serverless
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/vermont/silver/tmp'

# Crear carpeta tmp si no existe
dbutils.fs.mkdirs('/Volumes/workspace/vermont/silver/tmp')
print("✓ Ruta temporal configurada")

In [0]:
# Limpiar nombres de columnas para SparkML
import re

def limpiar_col(nombre):
    # Reemplazar espacios y caracteres especiales por _
    return re.sub(r'[ ,;{}()\n\t=áéíóúÁÉÍÓÚñÑ]', '_', nombre)

# Renombrar todas las columnas
nuevos_nombres = {c: limpiar_col(c) for c in df_weighted.columns}
for viejo, nuevo in nuevos_nombres.items():
    if viejo != nuevo:
        df_weighted = df_weighted.withColumnRenamed(viejo, nuevo)

# Actualizar lista de FEATURES con nombres limpios
FEATURES_CLEAN = [limpiar_col(f) for f in FEATURES]

print("✓ Columnas renombradas")
print(f"Features actualizadas:")
for f in FEATURES_CLEAN:
    print(f"  {f}")

In [0]:
# CELDA 2 — Modelo con class weights y cross-validation
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# Convertir target a índice numérico
indexer = StringIndexer(
    inputCol="nivel_riesgo",
    outputCol="label",
    stringOrderType="alphabetAsc"
)

# Calcular class weights para balanceo
conteos = df_weighted.groupBy('nivel_riesgo').count().toPandas()
n_total = df_weighted.count()
n_clases = 3

pesos = {}
for _, row in conteos.iterrows():
    pesos[row['nivel_riesgo']] = round(n_total / (n_clases * row['count']), 4)

print("Class weights calculados:")
for k, v in pesos.items():
    print(f"  {k}: {v}")

# Ensamblar features con nombres limpios
assembler = VectorAssembler(
    inputCols=FEATURES_CLEAN,
    outputCol="features"
)

# Random Forest con weightCol
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=5,
    seed=42
)

# Pipeline
pipeline = Pipeline(stages=[indexer, assembler, rf])

# Cross-validation k=5
paramGrid = ParamGridBuilder()\
    .addGrid(rf.numTrees, [50, 100])\
    .addGrid(rf.maxDepth, [3, 5])\
    .build()

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

# Entrenar
print("\nEntrenando con 5-Fold Cross-Validation...")
cv_modelo = cv.fit(df_weighted)
print("✓ Entrenamiento completo")

# Mejor modelo
mejor_modelo = cv_modelo.bestModel
rf_model = mejor_modelo.stages[-1]
print(f"\nMejores hiperparámetros:")
print(f"  numTrees: {rf_model.getNumTrees}")
print(f"  maxDepth: {rf_model.getOrDefault('maxDepth')}")

# Métricas
print(f"\n=== MÉTRICAS CROSS-VALIDATION (k=5) ===")
print(f"F1-Score promedio: {max(cv_modelo.avgMetrics):.4f}")

In [0]:
# CELDA 3 — Importancia de variables y guardar modelo
import pandas as pd
import matplotlib.pyplot as plt

# Extraer importancia de features del mejor modelo
rf_best = cv_modelo.bestModel.stages[-1]
importancias = pd.DataFrame({
    'feature': FEATURES_CLEAN,
    'importancia': rf_best.featureImportances.toArray()
}).sort_values('importancia', ascending=False)

print("=== IMPORTANCIA DE VARIABLES ===")
print(importancias.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#e74c3c' if i < 3 else '#3498db' for i in range(len(importancias))]
ax.barh(importancias['feature'][::-1], 
        importancias['importancia'][::-1],
        color=colores[::-1])
ax.set_title('Importancia de variables — Random Forest\nVermont School 2025-26', 
             fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('/tmp/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado")

# Predicciones sobre todos los datos
predicciones = cv_modelo.bestModel.transform(df_weighted)

# Guardar predicciones en Silver
predicciones.select(
    'id_estudiante', 'seccion_anon', 'grado',
    'nivel_riesgo', 'label', 'prediction',
    'probability'
).write.mode("overwrite").parquet(f"{SILVER}/modelo_predicciones")

# Guardar modelo
cv_modelo.bestModel.write().overwrite().save(
    "/Volumes/workspace/vermont/silver/modelo_rf"
)

print(f"✓ Predicciones guardadas en Silver")
print(f"✓ Modelo guardado en Silver")